# Data Center Cooling: Three Predictive Regression Models

This Colab notebook turns three of the five research questions into predictive models:

1. **Logistic regression:** Will the controller choose `Increase Chiller`?
2. **Linear regression:** What will the rack airflow temperature increase be?
3. **Linear regression:** What will the total cooling energy cost be?

The data is sorted by timestamp. The first 80% is used for training and the final 20% for testing, which tests the models on later observations.

## 1. Connect Google Drive and load the dataset

Run this cell first. Change `CSV_PATH` only if the dataset has moved.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
CSV_PATH = '/content/drive/MyDrive/ByteSmart Internship - Ranveer Singh/cold_source_control_dataset.csv'

df = pd.read_csv(CSV_PATH)
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.sort_values('Timestamp').reset_index(drop=True)
df['Thermal_Delta(°C)'] = df['Outlet_Temperature(°C)'] - df['Inlet_Temperature(°C)']
df['Increase_Chiller'] = (df['Cooling_Strategy_Action'] == 'Increase Chiller').astype(int)

split_index = int(len(df) * 0.80)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print(f'Dataset rows: {len(df):,}')
print(f'Training rows: {len(train_df):,}')
print(f'Testing rows: {len(test_df):,}')
display(df.head())

## 2. Logistic regression: predict `Increase Chiller`

**Predictive question:** Based on workload and temperature conditions, can we predict whether the cooling controller will select `Increase Chiller`?

Current chiller and AHU usage are deliberately excluded. Including them could reveal the action after it has already happened, which would cause target leakage.

In [ ]:
logistic_features = [
    'Server_Workload(%)',
    'Inlet_Temperature(°C)',
    'Outlet_Temperature(°C)',
    'Ambient_Temperature(°C)',
    'Temperature_Deviation(°C)'
]

X_train_log = train_df[logistic_features]
X_test_log = test_df[logistic_features]
y_train_log = train_df['Increase_Chiller']
y_test_log = test_df['Increase_Chiller']

if y_train_log.nunique() < 2:
    raise ValueError('The training period has only one class. Use more data or change the split.')

logistic_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=2000))
])
logistic_model.fit(X_train_log, y_train_log)
logistic_predictions = logistic_model.predict(X_test_log)
logistic_probabilities = logistic_model.predict_proba(X_test_log)[:, 1]

print(f'Training Increase Chiller rate: {y_train_log.mean():.2%}')
print(f'Testing Increase Chiller rate:  {y_test_log.mean():.2%}\n')
print(classification_report(
    y_test_log, logistic_predictions, labels=[0, 1],
    target_names=['Other action', 'Increase Chiller'], zero_division=0
))
if y_test_log.nunique() == 2:
    print(f'ROC AUC: {roc_auc_score(y_test_log, logistic_probabilities):.3f}')
else:
    print('ROC AUC is unavailable because the test period contains one class.')

ConfusionMatrixDisplay.from_predictions(
    y_test_log, logistic_predictions,
    display_labels=['Other', 'Increase Chiller'], cmap='Blues'
)
plt.title('Increase Chiller: Actual vs. Predicted')
plt.show()

logistic_coefficients = pd.Series(
    logistic_model.named_steps['model'].coef_[0], index=logistic_features
).sort_values(key=abs, ascending=False)
display(logistic_coefficients.rename('Standardized coefficient').to_frame())

## 3. Linear regression: predict the thermal delta

**Predictive question:** Based on workload, ambient conditions, inlet temperature, and cooling utilization, can we predict the outlet-minus-inlet temperature increase across the rack airflow path?

In [ ]:
thermal_features = [
    'Server_Workload(%)',
    'Ambient_Temperature(°C)',
    'Inlet_Temperature(°C)',
    'Chiller_Usage(%)',
    'AHU_Usage(%)'
]
thermal_target = 'Thermal_Delta(°C)'

thermal_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
thermal_model.fit(train_df[thermal_features], train_df[thermal_target])
thermal_predictions = thermal_model.predict(test_df[thermal_features])

thermal_mae = mean_absolute_error(test_df[thermal_target], thermal_predictions)
thermal_rmse = np.sqrt(mean_squared_error(test_df[thermal_target], thermal_predictions))
thermal_r2 = r2_score(test_df[thermal_target], thermal_predictions)
print(f'MAE:  {thermal_mae:.3f} °C')
print(f'RMSE: {thermal_rmse:.3f} °C')
print(f'R²:   {thermal_r2:.3f}')

plt.figure(figsize=(7, 6))
plt.scatter(test_df[thermal_target], thermal_predictions, alpha=0.45)
limits = [
    min(test_df[thermal_target].min(), thermal_predictions.min()),
    max(test_df[thermal_target].max(), thermal_predictions.max())
]
plt.plot(limits, limits, 'r--', label='Perfect prediction')
plt.xlabel('Actual thermal delta (°C)')
plt.ylabel('Predicted thermal delta (°C)')
plt.title('Thermal Delta: Actual vs. Predicted')
plt.legend()
plt.show()

thermal_coefficients = pd.Series(
    thermal_model.named_steps['model'].coef_, index=thermal_features
).sort_values(key=abs, ascending=False)
display(thermal_coefficients.rename('Standardized coefficient').to_frame())

## 4. Linear regression: predict total energy cost

**Predictive question:** Based on workload, temperatures, and cooling utilization, can we predict total cooling energy cost?

Cooling unit power is excluded because energy cost may be calculated directly from power. Leaving it out makes this an earlier and more useful operating-condition forecast instead of a disguised cost calculation.

In [ ]:
cost_features = [
    'Server_Workload(%)',
    'Ambient_Temperature(°C)',
    'Inlet_Temperature(°C)',
    'Outlet_Temperature(°C)',
    'Chiller_Usage(%)',
    'AHU_Usage(%)',
    'Temperature_Deviation(°C)'
]
cost_target = 'Total_Energy_Cost($)'

cost_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
cost_model.fit(train_df[cost_features], train_df[cost_target])
cost_predictions = cost_model.predict(test_df[cost_features])

cost_mae = mean_absolute_error(test_df[cost_target], cost_predictions)
cost_rmse = np.sqrt(mean_squared_error(test_df[cost_target], cost_predictions))
cost_r2 = r2_score(test_df[cost_target], cost_predictions)
print(f'MAE:  ${cost_mae:.4f}')
print(f'RMSE: ${cost_rmse:.4f}')
print(f'R²:   {cost_r2:.3f}')

plt.figure(figsize=(7, 6))
plt.scatter(test_df[cost_target], cost_predictions, alpha=0.45, color='teal')
limits = [
    min(test_df[cost_target].min(), cost_predictions.min()),
    max(test_df[cost_target].max(), cost_predictions.max())
]
plt.plot(limits, limits, 'r--', label='Perfect prediction')
plt.xlabel('Actual total energy cost ($)')
plt.ylabel('Predicted total energy cost ($)')
plt.title('Energy Cost: Actual vs. Predicted')
plt.legend()
plt.show()

cost_coefficients = pd.Series(
    cost_model.named_steps['model'].coef_, index=cost_features
).sort_values(key=abs, ascending=False)
display(cost_coefficients.rename('Standardized coefficient').to_frame())

## 5. Compare and export the test predictions

In [ ]:
results = test_df[['Timestamp', 'Cooling_Strategy_Action', 'Thermal_Delta(°C)', 'Total_Energy_Cost($)']].copy()
results['Probability_Increase_Chiller'] = logistic_probabilities
results['Predicted_Increase_Chiller'] = logistic_predictions
results['Predicted_Thermal_Delta(°C)'] = thermal_predictions
results['Predicted_Total_Energy_Cost($)'] = cost_predictions

OUTPUT_PATH = '/content/drive/MyDrive/ByteSmart Internship - Ranveer Singh/three_regression_predictions.csv'
results.to_csv(OUTPUT_PATH, index=False)
print(f'Saved predictions to: {OUTPUT_PATH}')
display(results.head(10))